<a href="https://colab.research.google.com/github/Nurfinkaa/MachineLearning/blob/main/js6/TG6_4_ANN_244107020211_NURFINKA_LAILASARI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Praktikum 4**

In [1]:
!pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520034 sha256=ce47b6e8e34e392bf4e9be1c278fc1b5d21381bb92912c0c4ae7b4c3fcbc02e4
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy


In [2]:
!pip install faiss-cpu hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 24.9 MB/s eta 0:00:00
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679796 sha256=11c5d464358af7acb283f5c4c08e460638c5e7514c1c44d32f0706b3f5df3fee
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [4]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 22.107032537460327 detik
Query time: 0.0002105236053466797 detik
Neighbors: [288128, 326271, 657221, 317393, 908642] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.01374506950378418 detik
Query time: 0.012267827987670898 detik
Neighbors: [288128 326271 657221 317393 908642] ...

=== HNSW (hnswlib) ===
Build time: 163.95053696632385 detik
Query time: 0.00023031234741210938 detik
Neighbors: [288128 326271 657221 317393 908642] ...


**Lakukan percobaan pada metric distance yang berbeda. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.**

In [3]:
import numpy as np
import time
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex

In [5]:
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM atau waktu terbatas
dim = 5
k = 10
n_query = 50
seed = 42

rng = np.random.default_rng(seed)
X = rng.random((n_data, dim)).astype(np.float32)
Q = rng.random((n_query, dim)).astype(np.float32)

# nama metric -> (metric Annoy, metric HNSW)
METRICS = {
    "Euclidean":     ("euclidean", "l2"),
    "Cosine":        ("angular",   "cosine"),
    "Inner Product": ("dot",       "ip"),
}

def avg_query_ms(fn):
    t0 = time.perf_counter()
    out = [fn(i) for i in range(len(Q))]
    return (time.perf_counter() - t0) / len(Q) * 1000, out

rows = []
for nama, (m_annoy, m_hnsw) in METRICS.items():
    print("=== Metric:", nama, flush=True)

    # ---------- FAISS Flat (exact, dijadikan ground truth) ----------
    Xf, Qf = X.copy(), Q.copy()
    if nama == "Euclidean":
        fi = faiss.IndexFlatL2(dim)
    else:
        fi = faiss.IndexFlatIP(dim)
        if nama == "Cosine":                 # cosine = inner product pada vektor ternormalisasi
            faiss.normalize_L2(Xf)
            faiss.normalize_L2(Qf)
    t0 = time.perf_counter()
    fi.add(Xf)
    t_build = time.perf_counter() - t0
    qt, truth = avg_query_ms(lambda i: fi.search(Qf[i:i+1], k)[1][0])
    rows.append({"Metric": nama, "Model": "FAISS Flat (exact)", "Build (s)": round(t_build, 3),
                 "Query (ms)": round(qt, 4), "Recall@10": 1.0,
                 "Neighbors (5 teratas)": list(map(int, truth[0][:5]))})

    # ---------- Annoy (10 tree) ----------
    t0 = time.perf_counter()
    ai = AnnoyIndex(dim, m_annoy)
    ai.set_seed(seed)
    for i in range(n_data):
        ai.add_item(i, X[i])
    ai.build(10)
    t_build = time.perf_counter() - t0
    qt, out = avg_query_ms(lambda i: ai.get_nns_by_vector(Q[i], k))
    rec = np.mean([len(set(a) & set(t)) / k for a, t in zip(out, truth)])
    rows.append({"Metric": nama, "Model": "Annoy (10 tree)", "Build (s)": round(t_build, 3),
                 "Query (ms)": round(qt, 4), "Recall@10": round(float(rec), 2),
                 "Neighbors (5 teratas)": list(map(int, out[0][:5]))})
    del ai

    # ---------- HNSW (hnswlib) ----------
    t0 = time.perf_counter()
    hi = hnswlib.Index(space=m_hnsw, dim=dim)
    hi.init_index(max_elements=n_data, ef_construction=200, M=16, random_seed=seed)
    hi.add_items(X)
    t_build = time.perf_counter() - t0
    hi.set_ef(50)
    qt, out = avg_query_ms(lambda i: hi.knn_query(Q[i:i+1], k=k)[0][0])
    rec = np.mean([len(set(a) & set(t)) / k for a, t in zip(out, truth)])
    rows.append({"Metric": nama, "Model": "HNSW (M=16, ef=50)", "Build (s)": round(t_build, 3),
                 "Query (ms)": round(qt, 4), "Recall@10": round(float(rec), 2),
                 "Neighbors (5 teratas)": list(map(int, out[0][:5]))})
    del hi

=== Metric: Euclidean
=== Metric: Cosine
=== Metric: Inner Product


In [6]:
# ---- Cetak tabel ----
df = pd.DataFrame(rows)
print("\n" + "=" * 130)
print(f"{'Metric':<15}{'Model':<22}{'Build (s)':<12}{'Query (ms)':<13}{'Recall@10':<11}{'Neighbors (5 teratas)'}")
print("=" * 130)
for _, r in df.iterrows():
    print(f"{r['Metric']:<15}{r['Model']:<22}{r['Build (s)']:<12}{r['Query (ms)']:<13}{r['Recall@10']:<11}{r['Neighbors (5 teratas)']}")


Metric         Model                 Build (s)   Query (ms)   Recall@10  Neighbors (5 teratas)
Euclidean      FAISS Flat (exact)    0.014       12.3458      1.0        [28012, 827330, 460931, 199693, 783903]
Euclidean      Annoy (10 tree)       19.607      0.0609       0.95       [28012, 827330, 460931, 199693, 783903]
Euclidean      HNSW (M=16, ef=50)    160.157     0.0637       1.0        [28012, 827330, 460931, 199693, 783903]
Cosine         FAISS Flat (exact)    0.005       4.422        1.0        [7535, 407763, 28012, 633452, 320155]
Cosine         Annoy (10 tree)       27.297      0.0603       0.97       [7535, 407763, 28012, 633452, 320155]
Cosine         HNSW (M=16, ef=50)    150.129     0.0523       1.0        [7535, 407763, 28012, 633452, 320155]
Inner Product  FAISS Flat (exact)    0.005       4.5748       1.0        [755592, 475499, 519344, 20442, 784282]
Inner Product  Annoy (10 tree)       44.37       0.0286       0.0        [140134, 188639, 416620, 545309, 733172]
Inner